In [2]:
import pandas as pd
import numpy as np

In [3]:
df = pd.read_csv("C:/Users/anasz/Desktop/uni bitch/FYP shi/data/Automated_Traffic_Volume_Counts_20260928.csv")

In [4]:
import os
folder = r"C:/Users/anasz/Desktop/uni bitch/FYP shi/data"
print([f for f in os.listdir(folder) if f.lower().endswith(".csv")])

['Automated_Traffic_Volume_Counts_20260928.csv', 'Traffic_Volume_Counts_(Historical)_20260926.csv']


In [5]:
hist_path = r"C:/Users/anasz/Desktop/uni bitch/FYP shi/data/Traffic_Volume_Counts_(Historical)_20260926.csv"  
hist = pd.read_csv(hist_path)

assert hist.shape[1] == 31, f"Wrong file: expected 31 columns, got {hist.shape[1]}. Columns: {hist.columns[:5].tolist()}"

hour_cols = [f"h{i:02d}" for i in range(24)]
hist.columns = list(hist.columns[:7]) + hour_cols

for c in hour_cols:
    hist[c] = pd.to_numeric(hist[c].astype(str).str.replace(",", "", regex=False), errors="coerce")

hist["Date"] = pd.to_datetime(hist["Date"], format="%m/%d/%Y", errors="coerce")

print("hist:", hist.shape, "| dates:", hist["Date"].min(), "->", hist["Date"].max())
print("Unparsed dates:", hist["Date"].isna().sum())
print("Hour dtypes:", hist[hour_cols].dtypes.unique())

hist: (42756, 31) | dates: 2012-01-08 00:00:00 -> 2021-05-09 00:00:00
Unparsed dates: 0
Hour dtypes: [dtype('float64')]


In [6]:
df.info()
df.isna().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1875154 entries, 0 to 1875153
Data columns (total 14 columns):
 #   Column     Dtype 
---  ------     ----- 
 0   RequestID  int64 
 1   Boro       object
 2   Yr         int64 
 3   M          int64 
 4   D          int64 
 5   HH         int64 
 6   MM         int64 
 7   Vol        object
 8   SegmentID  int64 
 9   WktGeom    object
 10  street     object
 11  fromSt     object
 12  toSt       object
 13  Direction  object
dtypes: int64(7), object(7)
memory usage: 200.3+ MB


RequestID       0
Boro            0
Yr              0
M               0
D               0
HH              0
MM              0
Vol             0
SegmentID       0
WktGeom         0
street          0
fromSt          0
toSt         1246
Direction       0
dtype: int64

In [7]:
df = df.sort_values(by= "Yr")
df.head(-10)

,RequestID,Boro,Yr,M,D,HH,MM,Vol,SegmentID,WktGeom,street,fromSt,toSt,Direction
1383250,2276,Manhattan,2000,1,7,13,15,189,32956,POINT (986884.7 207042.2),W/B UNION SQUARE E @ E 14 ST,E 14 ST/BROADWAY,4 AV,WB
1383251,2276,Manhattan,2000,1,7,13,30,226,32956,POINT (986884.7 207042.2),W/B UNION SQUARE E @ E 14 ST,E 14 ST/BROADWAY,4 AV,WB
1371884,2276,Manhattan,2000,1,5,2,15,205,32956,POINT (986884.7 207042.2),W/B UNION SQUARE E @ E 14 ST,E 14 ST/BROADWAY,4 AV,WB
1371885,2276,Manhattan,2000,1,5,2,30,215,32956,POINT (986884.7 207042.2),W/B UNION SQUARE E @ E 14 ST,E 14 ST/BROADWAY,4 AV,WB
1371886,2276,Manhattan,2000,1,5,2,45,234,32956,POINT (986884.7 207042.2),W/B UNION SQUARE E @ E 14 ST,E 14 ST/BROADWAY,4 AV,WB
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1532292,39920,Bronx,2026,1,27,15,15,127,79166,POINT (1016065.8049035381 247299.03220239395),SOUTHERN BOULEVARD,Marmion Avenue,East 180 Street,SB
1532293,39920,Bronx,2026,1,27,15,30,122,79166,POINT (1016065.8049035381 247299.03220239395),SOUTHERN BOULEVARD,Marmion Avenue,East 180 Street,SB
1532328,39920,Bronx,2026,1,28,0,15,25,79166,POINT (1016065.8049035381 247299.03220239395),SOUTHERN BOULEVARD,Marmion Avenue,East 180 Street,SB
1532314,39920,Bronx,2026,1,27,20,45,79,79166,POINT (1016065.8049035381 247299.03220239395),SOUTHERN BOULEVARD,Marmion Avenue,East 180 Street,SB


In [9]:
df["SegmentID"].unique()

array([ 32956, 138673,  73031, ..., 149299, 117844, 110430])

In [10]:
a = df.copy()

# Vol: convert and check for junk
raw_vol = df["Vol"].astype(str)
a["Vol"] = pd.to_numeric(raw_vol.str.replace(",", "", regex=False), errors="coerce")

# One timestamp from the 5 columns
a["ts"] = pd.to_datetime(
    dict(year=a["Yr"], month=a["M"], day=a["D"], hour=a["HH"], minute=a["MM"]),
    errors="coerce")
a["date"] = a["ts"].dt.normalize()

In [11]:
print("Junk (non-numeric) Vol:", a["Vol"].isna().sum())
print(raw_vol[a["Vol"].isna()].value_counts().head(10))
print("Cells with commas:", raw_vol.str.contains(",", regex=False).sum())
print(a["Vol"].describe(percentiles=[.01, .05, .5, .95, .99, .999]))
print("Zeros:", (a["Vol"] == 0).sum(), "| Negatives:", (a["Vol"] < 0).sum())

Junk (non-numeric) Vol: 0
Series([], Name: count, dtype: int64)
Cells with commas: 15257
count    1.875154e+06
mean     1.137309e+02
std      1.904752e+02
min     -1.000000e+00
1%       0.000000e+00
5%       0.000000e+00
50%      5.900000e+01
95%      3.990000e+02
99%      9.420000e+02
99.9%    1.700847e+03
max      5.425000e+03
Name: Vol, dtype: float64
Zeros: 96278 | Negatives: 1


In [12]:
print("Unparseable timestamps:", a["ts"].isna().sum())
print("Range:", a["ts"].min(), "->", a["ts"].max())
print(a["Yr"].value_counts().sort_index())
print("MM values:", a["MM"].value_counts().sort_index().to_dict())
print("HH values:", sorted(a["HH"].unique()))
print(a["ts"].dt.day_name().value_counts())

Unparseable timestamps: 0
Range: 2000-01-01 00:15:00 -> 2026-02-03 23:45:00
Yr
2000      1904
2006       664
2007      8130
2008     32482
2009    122851
2010    132016
2011    120249
2012    128222
2013    128762
2014    130754
2015    124428
2016    129617
2017    123901
2018     98961
2019    115730
2020     40224
2021     85810
2022     78652
2023     85106
2024     95203
2025     82368
2026      9120
Name: count, dtype: int64
MM values: {0: 473659, 10: 718, 15: 466215, 20: 718, 30: 466210, 40: 718, 45: 466199, 50: 717}
HH values: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18), np.int64(19), np.int64(20), np.int64(21), np.int64(22), np.int64(23)]
ts
Tuesday      272287
Wednesday    271707
Thursday     271243
Monday       266587
Saturday     264890
Sunday       264877
Friday   

In [13]:
g = a.groupby(["SegmentID", "Direction", "date"])["ts"]
first = g.min().dt.strftime("%H:%M")
last = g.max().dt.strftime("%H:%M")
print("Earliest timestamp of each day:\n", first.value_counts().head(8))
print("Latest timestamp of each day:\n", last.value_counts().head(8))

Earliest timestamp of each day:
 ts
00:00    19673
10:00      151
07:00      147
00:15      147
08:00      143
09:45      137
09:00      137
08:30      128
Name: count, dtype: int64
Latest timestamp of each day:
 ts
23:45    19374
23:00      446
09:45      144
09:30      144
09:00      139
08:15      136
09:15      136
08:00      135
Name: count, dtype: int64


In [14]:
k1 = ["SegmentID", "Direction", "ts"]
print("Full-row duplicates:", a.duplicated().sum())
print("Duplicate (SegmentID, Direction, ts):", a.duplicated(k1).sum())
print("Duplicate (RequestID + key):", a.duplicated(["RequestID"] + k1).sum())

d = a[a.duplicated(k1, keep=False)].sort_values(k1)
print("Rows in duplicate groups:", len(d))
print(d["Yr"].value_counts().sort_index())
d.head(10)

Full-row duplicates: 0
Duplicate (SegmentID, Direction, ts): 5146
Duplicate (RequestID + key): 5146
Rows in duplicate groups: 10292
Yr
2023    9534
2024     758
Name: count, dtype: int64


,RequestID,Boro,Yr,M,D,HH,MM,Vol,SegmentID,WktGeom,street,fromSt,toSt,Direction,ts,date
557807,36401,Brooklyn,2023,6,19,0,0,18,50963,POINT (1022726.375963018 184975.11727740432),SUTTER AVENUE,South Conduit Avenue,Dead end,WB,2023-06-19 00:00:00,2023-06-19
114037,36401,Brooklyn,2023,6,19,0,0,57,50963,POINT (1022726.375963018 184975.11727740432),SUTTER AVENUE,South Conduit Avenue,Dead end,WB,2023-06-19 00:00:00,2023-06-19
557808,36401,Brooklyn,2023,6,19,0,15,34,50963,POINT (1022726.375963018 184975.11727740432),SUTTER AVENUE,South Conduit Avenue,Dead end,WB,2023-06-19 00:15:00,2023-06-19
114038,36401,Brooklyn,2023,6,19,0,15,36,50963,POINT (1022726.375963018 184975.11727740432),SUTTER AVENUE,South Conduit Avenue,Dead end,WB,2023-06-19 00:15:00,2023-06-19
557809,36401,Brooklyn,2023,6,19,0,30,14,50963,POINT (1022726.375963018 184975.11727740432),SUTTER AVENUE,South Conduit Avenue,Dead end,WB,2023-06-19 00:30:00,2023-06-19
114039,36401,Brooklyn,2023,6,19,0,30,52,50963,POINT (1022726.375963018 184975.11727740432),SUTTER AVENUE,South Conduit Avenue,Dead end,WB,2023-06-19 00:30:00,2023-06-19
557810,36401,Brooklyn,2023,6,19,0,45,22,50963,POINT (1022726.375963018 184975.11727740432),SUTTER AVENUE,South Conduit Avenue,Dead end,WB,2023-06-19 00:45:00,2023-06-19
114040,36401,Brooklyn,2023,6,19,0,45,46,50963,POINT (1022726.375963018 184975.11727740432),SUTTER AVENUE,South Conduit Avenue,Dead end,WB,2023-06-19 00:45:00,2023-06-19
557811,36401,Brooklyn,2023,6,19,1,0,24,50963,POINT (1022726.375963018 184975.11727740432),SUTTER AVENUE,South Conduit Avenue,Dead end,WB,2023-06-19 01:00:00,2023-06-19
114041,36401,Brooklyn,2023,6,19,1,0,33,50963,POINT (1022726.375963018 184975.11727740432),SUTTER AVENUE,South Conduit Avenue,Dead end,WB,2023-06-19 01:00:00,2023-06-19


In [15]:
print(a["Direction"].value_counts(dropna=False))
print(a["Boro"].value_counts(dropna=False))

Direction
NB    474645
SB    469877
EB    463384
WB    460720
EW      3564
NS      2964
Name: count, dtype: int64
Boro
Queens           560928
Brooklyn         540695
Manhattan        343555
Bronx            308124
Staten Island    121852
Name: count, dtype: int64


In [16]:
n = a.groupby(["SegmentID", "Direction", "date"])["ts"].nunique()
print(n.describe())
print(n.value_counts().head(15))
print("Days with exactly 96 intervals:", (n == 96).sum(), "of", len(n))

count    23366.000000
mean        80.031156
std         29.497449
min          1.000000
25%         73.000000
50%         96.000000
75%         96.000000
max         96.000000
Name: ts, dtype: float64
ts
96    17265
24      411
1       351
2       291
3       197
4       164
48      139
72      119
5       113
40      106
30      101
56      100
6        96
57       96
31       95
Name: count, dtype: int64
Days with exactly 96 intervals: 17265 of 23366


In [17]:
g = a.groupby("SegmentID")
print("SegmentIDs:", a["SegmentID"].nunique())
print("Segments with >1 street:", (g["street"].nunique() > 1).sum())
print("Segments with >1 fromSt:", (g["fromSt"].nunique() > 1).sum())
print("Segments with >1 geometry:", (g["WktGeom"].nunique() > 1).sum())
print("RequestIDs:", a["RequestID"].nunique())
print("RequestIDs with >1 SegmentID:", (a.groupby("RequestID")["SegmentID"].nunique() > 1).sum())
print("SegmentIDs with >1 RequestID:", (g["RequestID"].nunique() > 1).sum())

multi = g["street"].nunique()
ex = multi[multi > 1].index[:6]
(a[a["SegmentID"].isin(ex)][["SegmentID", "street", "fromSt", "toSt", "Yr"]]
   .drop_duplicates().sort_values(["SegmentID", "Yr"]).head(30))

SegmentIDs: 3391
Segments with >1 street: 260
Segments with >1 fromSt: 200
Segments with >1 geometry: 261
RequestIDs: 2236
RequestIDs with >1 SegmentID: 585
SegmentIDs with >1 RequestID: 478


,SegmentID,street,fromSt,toSt,Yr
586602,2668,LUTEN AVE NB AT DEISUS ST,DEISIUS ST,EYLANDT ST,2009
586628,2668,S/B LUTEN AVE @ EYLANDT ST,DEISIUS ST,EYLANDT ST,2010
1802926,2789,LUTEN AVE NB AT EYLANDT ST,JANSEN ST,EYLANDT ST,2009
1820779,2789,LUTEN AVE SB AT JANSEN ST,JANSEN ST,EYLANDT ST,2009
1809931,2789,N/B LUTEN AVE @ EYLANDT ST,JANSEN ST,EYLANDT ST,2010
399425,6882,E/B AMBOY RD @ AINSWORTH AVE,CRANFORD ST,AINSWORTH AV/S RAILROAD AV,2009
399428,6882,AMBOY ROAD,Cranford Street,Ainsworth Avenue,2015
1861352,7729,N/B KELLY BLVD @ TRAVIS AVE,KLONDIKE AV,TRAVIS AV,2010
1861359,7729,KELLY BLVD,TRAVIS AV,KLONDIKE AV,2011
280747,8494,Forest Ave,Eunice Place,Simonson Ave,2012


In [18]:
coords = a["WktGeom"].str.extract(r"POINT \(([-\d\.]+) ([-\d\.]+)\)").astype(float)
coords.columns = ["x", "y"]
print("Unparsed geometry:", coords.isna().any(axis=1).sum())
print(a["WktGeom"].str.split(" ").str[0].value_counts())
print(coords.describe())

Unparsed geometry: 0
WktGeom
POINT    1875154
Name: count, dtype: int64
                  x             y
count  1.875154e+06  1.875154e+06
mean   1.006694e+06  2.025669e+05
std    2.641582e+04  3.227558e+04
min    9.147350e+05  1.207978e+05
25%    9.918645e+05  1.789068e+05
50%    1.005050e+06  2.008814e+05
75%    1.023201e+06  2.241365e+05
max    1.067177e+06  2.718439e+05


In [19]:
cov = (a.groupby("SegmentID")
         .agg(n_days=("date", "nunique"), n_years=("Yr", "nunique"),
              first=("ts", "min"), last=("ts", "max")))
print(cov[["n_days", "n_years"]].describe())
print("Segments with only 1 year:", (cov["n_years"] == 1).sum(), "of", len(cov))
print(cov["n_days"].value_counts().sort_index().head(15))

            n_days      n_years
count  3391.000000  3391.000000
mean      6.110292     1.191094
std       7.655656     0.690041
min       1.000000     1.000000
25%       1.000000     1.000000
50%       4.000000     1.000000
75%       8.000000     1.000000
max     107.000000    10.000000
Segments with only 1 year: 3021 of 3391
n_days
1     932
2     445
3     207
4     215
5     275
6     113
7     306
8     103
9     176
10    106
11     83
12     59
13     43
14     58
15     38
Name: count, dtype: int64


In [20]:
hs, as_ = set(hist["SegmentID"]), set(a["SegmentID"])
print("Historical segments:", len(hs), "| automated:", len(as_), "| shared:", len(hs & as_))

hd = hist[["SegmentID", "Date"]].drop_duplicates().rename(columns={"Date": "date"})
ad = a[["SegmentID", "date"]].drop_duplicates()
both = hd.merge(ad, on=["SegmentID", "date"])
print("Segment-days: historical", len(hd), "| automated", len(ad), "| in both", len(both))

print("Automated rows within the historical date range:",
      ((a["ts"] >= hist["Date"].min()) &
       (a["ts"] <= hist["Date"].max() + pd.Timedelta(days=1))).sum())

Historical segments: 1956 | automated: 3391 | shared: 346
Segment-days: historical 28487 | automated 20720 | in both 788
Automated rows within the historical date range: 1050245


In [21]:
shared = sorted(set(hist["SegmentID"]) & set(a["SegmentID"]))

def norm(s):
    return (s.astype(str).str.upper()
             .str.replace(r"[^A-Z0-9 ]", " ", regex=True)
             .str.replace(r"\bAVE\b", "AVENUE", regex=True)
             .str.replace(r"\bST\b", "STREET", regex=True)
             .str.replace(r"\bRD\b", "ROAD", regex=True)
             .str.replace(r"\bBLVD\b", "BOULEVARD", regex=True)
             .str.replace(r"\s+", " ", regex=True).str.strip())

hn = (hist[hist["SegmentID"].isin(shared)].assign(name=lambda x: norm(x["Roadway Name"]))
        .groupby("SegmentID")["name"].agg(set))
an = (a[a["SegmentID"].isin(shared)].assign(name=lambda x: norm(x["street"]))
        .groupby("SegmentID")["name"].agg(set))

def match(sid):
    return any(h_ in a_ or a_ in h_ for h_ in hn[sid] for a_ in an[sid])

res = pd.Series({sid: match(sid) for sid in shared})
print("Shared:", len(res), "| name consistent:", res.sum(), "| inconsistent:", (~res).sum())
for sid in res[~res].index[:15]:
    print(sid, "| hist:", list(hn[sid])[:2], "| auto:", list(an[sid])[:2])

Shared: 346 | name consistent: 318 | inconsistent: 28
26075 | hist: ['SHORE PKWY SOUTH'] | auto: ['SHORE PARKWAY']
27120 | hist: ['AVENUE N'] | auto: ['AV N']
31559 | hist: ['WEST BROADWAY'] | auto: ['W BROADWAY']
34170 | hist: ['WEST 39TH STREET', 'WEST 39 STREET'] | auto: ['W 39 STREET']
36023 | hist: ['5 AVENUE'] | auto: ['S B 5TH AVENUE E 60TH STREET']
36274 | hist: ['3 AVENUE'] | auto: ['N B 3RD AVENUE 2 LANES E SIDE E 61 STREET']
38837 | hist: ['MT MORRIS PARK WEST'] | auto: ['MOUNT MORRIS PARK W']
41720 | hist: ['AVENUE D'] | auto: ['AV D']
45165 | hist: ['GREEN AVENUE'] | auto: ['GREENE AVENUE', 'GREENE AV']
50822 | hist: ['SOUTH CONDUIT AVENUE'] | auto: ['E B S CONDUIT AVENUE BELMONT AVENUE']
56117 | hist: ['KEW GARDENS ROAD'] | auto: ['KEW GDNS ROAD']
69154 | hist: ['FREDERICK DOUGLASS BOULEVARD', '8 AVENUE'] | auto: ['FRED DOUGLASS BOULEVARD']
71729 | hist: ['SHERIDANA AVENUE'] | auto: ['SHERIDAN AV']
95270 | hist: ['LEXINGTON AVENUE'] | auto: ['HUTCHINSON RIVER PARKWAY']
10

In [22]:
# Volume agreement on shared segment-hours (automated summed to hourly)
a1 = a[a["Direction"].isin(["NB", "SB", "EB", "WB"]) & a["MM"].isin([0, 15, 30, 45])
       & a["SegmentID"].isin(shared)].copy()
a1["hour"] = a1["ts"].dt.hour
ah = (a1.groupby(["SegmentID", "Direction", "date", "hour"])
        .agg(vol=("Vol", "sum"), n=("Vol", "size")).reset_index())
ah = ah[ah["n"] == 4]

hl = (hist[hist["Direction"].isin(["NB", "SB", "EB", "WB"]) & hist["SegmentID"].isin(shared)]
        .melt(id_vars=["SegmentID", "Direction", "Date"], value_vars=hour_cols,
              var_name="hr", value_name="hvol")
        .dropna(subset=["hvol"]))
hl["hour"] = hl["hr"].str[1:].astype(int)
hl = hl.rename(columns={"Date": "date"})

m = ah.merge(hl, on=["SegmentID", "Direction", "date", "hour"])
print("Matching segment-direction-hours:", len(m))
if len(m):
    m["ratio"] = m["vol"] / m["hvol"].replace(0, np.nan)
    print(m["ratio"].describe(percentiles=[.05, .25, .5, .75, .95]))
    print("Correlation:", m[["vol", "hvol"]].corr().iloc[0, 1])
    print("Within +-10%:", ((m["ratio"] - 1).abs() <= 0.10).mean())

Matching segment-direction-hours: 22275
count    22135.000000
mean         1.000953
std          0.043041
min          0.328244
5%           1.000000
25%          1.000000
50%          1.000000
75%          1.000000
95%          1.000000
max          3.046512
Name: ratio, dtype: float64
Correlation: 0.9988195945125063
Within +-10%: 0.9892255892255892


In [23]:
k = ["SegmentID", "Direction", "ts"]
dd = a[a.duplicated(k, keep=False)].copy()
dd["r"] = dd.groupby(k).cumcount()
w = dd.pivot_table(index=k, columns="r", values="Vol")

print("Groups:", len(w), "| groups with >2 rows:", (dd.groupby(k).size() > 2).sum())
print("Pairs with identical Vol:", (w[0] == w[1]).sum())
print("Correlation between copies:", w[[0, 1]].corr().iloc[0, 1])

tot = dd.groupby(["RequestID", "SegmentID", "Direction", "date", "r"])["Vol"].sum().unstack("r")
print((tot[1] / tot[0].replace(0, np.nan)).describe())
print("RequestIDs:", dd["RequestID"].nunique(), "| segments:", dd["SegmentID"].nunique())
print(dd.reset_index().groupby("r")["index"].agg(["min", "max"]))

Groups: 5146 | groups with >2 rows: 0
Pairs with identical Vol: 0
Correlation between copies: 0.851871270223011
count    54.000000
mean      1.174601
std       0.803068
min       0.269043
25%       0.752091
50%       1.006096
75%       1.316576
max       4.358043
dtype: float64
RequestIDs: 3 | segments: 6
     min      max
r                
0   2100  1825803
1  13335  1792518


In [24]:
print(a.nlargest(10, "Vol")[["RequestID", "SegmentID", "Direction", "ts", "Vol", "street"]])
print(a[a["Vol"] < 0][["RequestID", "SegmentID", "Direction", "ts", "Vol"]])

a["zero"] = a["Vol"] == 0
print(a.groupby("Yr")["zero"].mean().round(3))
print(a.groupby(a["ts"].dt.hour)["zero"].mean().round(3))

s = a.sort_values(["SegmentID", "Direction", "ts"])
chg = ((s["zero"] != s["zero"].shift()) | (s["SegmentID"] != s["SegmentID"].shift())
       | (s["Direction"] != s["Direction"].shift()))
s["run"] = chg.cumsum()
runs = s[s["zero"]].groupby("run").size()
print(runs.describe())
print("Zero runs >= 8 intervals:", (runs >= 8).sum(), "| >= 32:", (runs >= 32).sum())
print("Zero cells inside runs >= 8:", runs[runs >= 8].sum(), "of", s["zero"].sum())

         RequestID  SegmentID Direction                  ts   Vol  street
1525755       7620      36524        SB 2007-10-23 08:00:00  5425  FDR DR
1525754       7620      36524        SB 2007-10-23 07:00:00  5409  FDR DR
413162        7620      36524        SB 2007-10-26 08:00:00  5191  FDR DR
1525778       7620      36524        SB 2007-10-24 07:00:00  5182  FDR DR
1525731       7620      36524        SB 2007-10-22 08:00:00  5106  FDR DR
403024        7620      36524        SB 2007-10-20 19:00:00  5103  FDR DR
1525730       7620      36524        SB 2007-10-22 07:00:00  5092  FDR DR
1525763       7620      36524        SB 2007-10-23 16:00:00  5087  FDR DR
1525812       7620      36524        SB 2007-10-25 17:00:00  5019  FDR DR
413161        7620      36524        SB 2007-10-26 07:00:00  5011  FDR DR
         RequestID  SegmentID Direction         ts  Vol
1367019      12501     145593        WB 2010-09-21   -1
Yr
2000    0.002
2006    0.000
2007    0.037
2008    0.119
2009    0.121
2

In [25]:
st = (a.groupby(["RequestID", "SegmentID", "Direction", "date"])["MM"]
        .agg(lambda x: tuple(sorted(x.unique()))))
print(st.value_counts().head(10))
print(a[a["MM"].isin([10, 20, 40, 50])]["Yr"].value_counts().sort_index())

MM
(0, 15, 30, 45)    21808
(0,)                 641
(15, 30, 45)         186
(30,)                 92
(45,)                 89
(0, 15)               82
(30, 45)              81
(15,)                 74
(15, 30)              69
(0, 45)               62
Name: count, dtype: int64
Yr
2022    2871
Name: count, dtype: int64


In [26]:
key = ["RequestID", "SegmentID", "ts"]
sub = a[a["Direction"].isin(["EB", "WB", "NB", "SB", "EW", "NS"])]
p = sub.pivot_table(index=key, columns="Direction", values="Vol", aggfunc="first")
p = p.reindex(columns=["EB", "WB", "NB", "SB", "EW", "NS"])

ew = p[p["EW"].notna() & p["EB"].notna() & p["WB"].notna()]
print("EW rows:", p["EW"].notna().sum(), "| with matching EB and WB:", len(ew))
print("EW / (EB+WB):")
print((ew["EW"] / (ew["EB"] + ew["WB"])).replace([np.inf, -np.inf], np.nan)
      .describe(percentiles=[.05, .25, .5, .75, .95]))

ns = p[p["NS"].notna() & p["NB"].notna() & p["SB"].notna()]
print("\nNS rows:", p["NS"].notna().sum(), "| with matching NB and SB:", len(ns))
print("NS / (NB+SB):")
print((ns["NS"] / (ns["NB"] + ns["SB"])).replace([np.inf, -np.inf], np.nan)
      .describe(percentiles=[.05, .25, .5, .75, .95]))

print("\nEW by year:", a.loc[a["Direction"] == "EW", "Yr"].value_counts().sort_index().to_dict())
print("NS by year:", a.loc[a["Direction"] == "NS", "Yr"].value_counts().sort_index().to_dict())

EW rows: 3564 | with matching EB and WB: 0
EW / (EB+WB):
count    0.0
mean     NaN
std      NaN
min      NaN
5%       NaN
25%      NaN
50%      NaN
75%      NaN
95%      NaN
max      NaN
dtype: float64

NS rows: 2964 | with matching NB and SB: 0
NS / (NB+SB):
count    0.0
mean     NaN
std      NaN
min      NaN
5%       NaN
25%      NaN
50%      NaN
75%      NaN
95%      NaN
max      NaN
dtype: float64

EW by year: {2012: 1643, 2017: 1921}
NS by year: {2016: 2016, 2017: 948}


In [27]:
H = hist[hour_cols]

# Length of the trailing zero run (zeros counted back from h23)
Z = (H == 0).values[:, ::-1]
hist["tz"] = np.cumprod(Z, axis=1).sum(axis=1)
print("Trailing zero run length:")
print(hist["tz"].value_counts().sort_index())
print("\nShare of rows with >=3 trailing zeros, by year:")
print(hist.groupby(hist["Date"].dt.year)["tz"].apply(lambda s: (s >= 3).mean()).round(3))

# Do trailing-zero rows cluster on specific dates (campaign-wide outage)?
pad = hist.assign(pad=hist["tz"] >= 3).groupby("Date")["pad"].agg(["mean", "size"])
print("\nDates where most rows are trailing-zero padded:")
print(pad[pad["mean"] > 0.5].sort_values("size", ascending=False).head(10))

# Interior zeros with busy hours on both sides and a busy segment median
seg_med = H.groupby(hist["SegmentID"]).transform("median")
prev, nxt = H.shift(1, axis=1), H.shift(-1, axis=1)
interior = (H == 0) & (prev > 100) & (nxt > 100) & (seg_med > 100)
print("\nIsolated zeros between busy hours on busy segments:", int(interior.sum().sum()))

Trailing zero run length:
tz
0     42273
1        35
2        21
3        20
4         4
5         9
6         9
7         8
8        14
9        25
10       20
11       59
12       28
13       66
14       88
15       46
16        7
17        3
18        2
19        3
20       11
22        2
24        3
Name: count, dtype: int64

Share of rows with >=3 trailing zeros, by year:
Date
2012    0.050
2013    0.001
2014    0.000
2015    0.002
2016    0.001
2017    0.000
2018    0.000
2019    0.000
2020    0.001
2021    0.000
Name: tz, dtype: float64

Dates where most rows are trailing-zero padded:
                mean  size
Date                      
2012-02-17  0.717172    99
2012-01-20  0.555556    63
2012-01-27  0.555556    63
2012-01-28  0.750000    28
2012-01-21  0.666667    27
2012-02-25  1.000000    16

Isolated zeros between busy hours on busy segments: 34


In [28]:
CSCL_PATH = r"C:/Users/anasz/Desktop/uni bitch/FYP shi/data/Centerline_20261009.csv"  # replace
cs = pd.read_csv(CSCL_PATH, low_memory=False)
print(cs.shape)
print([c for c in cs.columns if "ID" in c.upper() or "SEG" in c.upper()])
print(cs.columns.tolist())

(122311, 64)
['PHYSICALID', 'OBJECTID', 'Left side Block Face ID', 'Right Side Block Face ID', 'Segloc Status', 'Left Side Subset', 'Right Side Subset', 'Segment Length', 'Street Width', 'Street Width IRR', 'JOINID', 'BPHYS_ID', 'GlobalID', 'SEGMENT_TYPE', 'SEGMENT_TYPE_VALUE']
['the_geom', 'PHYSICALID', 'L_LOW_HN', 'L_HIGH_HN', 'R_LOW_HN', 'R_HIGH_HN', 'L_ZIP', 'R_ZIP', 'STATUS', 'BIKE_LANE', 'TRAFDIR', 'RW_TYPE', 'PRE_TYPE', 'POST_TYPE', 'OBJECTID', 'FCC', 'Left side Block Face ID', 'Right Side Block Face ID', 'Average Travel Time', 'Roadway Jurisdiction', 'NOMINALDIR', 'ACCESSIBLE', 'NONPED', 'Borough Code', 'Borough Indicator', 'Segloc Status', 'San District Inc', 'Left Side Subset', 'Right Side Subset', 'Continuous Parity Flag', 'Twisted Parity Flag', 'Posted Speed', 'Segment Length', 'Street Width', 'Street Width IRR', 'Special Disaster', 'Fire Lane', 'CREATED_DATE', 'MODIFIED_DATE', 'WITHIN_BNDY_DIST', 'TRUCK ROUTE TYPE', 'Collection Method', 'FROM_LEVEL_CODE', 'TO_LEVEL_CODE', 

In [30]:
# If needed: pip install pyproj
from pyproj import Transformer

geo = a[["SegmentID", "Boro", "WktGeom"]].drop_duplicates("SegmentID").copy()
xy = geo["WktGeom"].str.extract(r"POINT \(([-\d\.]+) ([-\d\.]+)\)").astype(float)
t = Transformer.from_crs("EPSG:2263", "EPSG:4326", always_xy=True)
geo["lon"], geo["lat"] = t.transform(xy[0].values, xy[1].values)

in_nyc = geo["lat"].between(40.45, 41.0) & geo["lon"].between(-74.3, -73.65)
print("Inside NYC bounds:", in_nyc.mean().round(4), "of", len(geo))
print(geo.loc[~in_nyc, ["SegmentID", "Boro", "lat", "lon"]].head(10))

print("\nMedian lat/lon by borough:")
print(geo.groupby("Boro")[["lat", "lon"]].median().round(4))

Inside NYC bounds: 1.0 of 3391
Empty DataFrame
Columns: [SegmentID, Boro, lat, lon]
Index: []

Median lat/lon by borough:
                   lat      lon
Boro                           
Bronx          40.8437 -73.8813
Brooklyn       40.6578 -73.9589
Manhattan      40.7705 -73.9726
Queens         40.7293 -73.8388
Staten Island  40.5959 -74.1187


In [31]:
a["hour"] = a["ts"].dt.floor("h")
ten_rids = a.loc[a["MM"].isin([10, 20, 40, 50]), "RequestID"].unique()
print("10-minute RequestIDs:", len(ten_rids), "| segments:",
      a.loc[a["RequestID"].isin(ten_rids), "SegmentID"].nunique())

ten = a[a["RequestID"].isin(ten_rids)]
print("\nMinute patterns per study-day:")
print(ten.groupby(["RequestID", "SegmentID", "Direction", "date"])["MM"]
         .agg(lambda x: tuple(sorted(x.unique()))).value_counts().head(8))

# Intervals per hour in those studies (expect 6 for true 10-minute)
print("\nIntervals per hour:")
print(ten.groupby(["RequestID", "SegmentID", "Direction", "hour"]).size().value_counts().head(8))

# Hourly totals: 10-minute studies vs 15-minute studies, same segment/direction/hour, 2022
sub22 = a[(a["Yr"] == 2022) & a["Direction"].isin(["NB", "SB", "EB", "WB"])]
hr = (sub22.groupby(["SegmentID", "Direction", "hour", "RequestID"])
           .agg(vol=("Vol", "sum"), n=("Vol", "size"), mm=("MM", lambda x: tuple(sorted(x.unique()))))
           .reset_index())
hr["kind"] = np.where(hr["RequestID"].isin(ten_rids), "10min",
              np.where(hr["n"] == 4, "15min", "other"))
t10 = hr[(hr["kind"] == "10min") & (hr["n"] == 6)]
t15 = hr[hr["kind"] == "15min"]
m = t10.merge(t15, on=["SegmentID", "Direction", "hour"], suffixes=("_10", "_15"))
print("\nMatched hours (10-min vs 15-min studies):", len(m))
if len(m):
    print((m["vol_10"] / m["vol_15"].replace(0, np.nan)).describe(percentiles=[.05, .25, .5, .75, .95]))

10-minute RequestIDs: 1 | segments: 26

Minute patterns per study-day:
MM
(0, 15, 30, 45)     176
(10, 20, 40, 50)     32
(0, 15, 30)           2
(0, 15)               2
(45,)                 1
(15, 30)              1
(0, 15, 45)           1
(0, 30, 45)           1
Name: count, dtype: int64

Intervals per hour:
4    4420
2      19
1      17
3      14
Name: count, dtype: int64

Matched hours (10-min vs 15-min studies): 0


In [32]:
for tw, parts in [("EW", ["EB", "WB"]), ("NS", ["NB", "SB"])]:
    d = a[a["Direction"].isin([tw] + parts)].copy()
    d["hr"] = d["ts"].dt.hour
    segs = (set(d.loc[d["Direction"] == tw, "SegmentID"])
            & set(d.loc[d["Direction"].isin(parts), "SegmentID"]))
    print(tw, "segments with both two-way and directional rows:", len(segs))
    if segs:
        med = (d[d["SegmentID"].isin(segs)]
               .groupby(["SegmentID", "Direction", "hr"])["Vol"].median()
               .unstack("Direction").reindex(columns=[tw] + parts))
        print((med[tw] / med[parts].sum(axis=1)).describe(percentiles=[.25, .5, .75]))

EW segments with both two-way and directional rows: 0
NS segments with both two-way and directional rows: 1
count    24.000000
mean      0.052857
std       0.051772
min       0.000000
25%       0.000000
50%       0.047109
75%       0.100071
max       0.146119
dtype: float64


In [33]:
hs, as_ = set(hist["SegmentID"]), set(a["SegmentID"])
for c in ["PHYSICALID", "OBJECTID", "JOINID", "BPHYS_ID"]:
    s = set(pd.to_numeric(cs[c], errors="coerce").dropna().astype("int64"))
    print(f"{c:11s} unique={len(s):7d} | historical match={len(hs & s):5d} ({len(hs & s)/len(hs):.1%})"
          f" | automated match={len(as_ & s):5d} ({len(as_ & s)/len(as_):.1%})")

PHYSICALID  unique=    800 | historical match=    2 (0.1%) | automated match=    4 (0.1%)
OBJECTID    unique=    816 | historical match=    1 (0.1%) | automated match=    2 (0.1%)
JOINID      unique=      0 | historical match=    0 (0.0%) | automated match=    0 (0.0%)
BPHYS_ID    unique=      0 | historical match=    0 (0.0%) | automated match=    0 (0.0%)


In [34]:
ID_COL = "PHYSICALID"   # change if another column matched better

cs["_id"] = pd.to_numeric(cs[ID_COL], errors="coerce")
cs1 = cs.dropna(subset=["_id"]).copy()
cs1["_id"] = cs1["_id"].astype("int64")
print("Duplicate ids in centerline:", cs1["_id"].duplicated().sum())

cs_name = norm(cs1.drop_duplicates("_id").set_index("_id")["Full Street Name"])

hn_all = hist.assign(n=norm(hist["Roadway Name"])).groupby("SegmentID")["n"].agg(set)
an_all = a.assign(n=norm(a["street"])).groupby("SegmentID")["n"].agg(set)

def agree(names_by_seg):
    out = {}
    for sid, names in names_by_seg.items():
        if sid in cs_name.index:
            c = cs_name[sid]
            out[sid] = any((c in n) or (n in c) for n in names)
    return pd.Series(out)

for label, nb in [("historical", hn_all), ("automated", an_all)]:
    r = agree(nb)
    print(f"{label}: in centerline {len(r)} of {len(nb)} | name agrees {int(r.sum())} ({r.mean():.1%})")

Duplicate ids in centerline: 0
historical: in centerline 2 of 1956 | name agrees 0 (0.0%)
automated: in centerline 4 of 3391 | name agrees 0 (0.0%)


In [35]:
cols = ["PHYSICALID", "OBJECTID", "JOINID", "BPHYS_ID"]
print(cs[cols].head(8))
print(cs[cols].dtypes)
print(cs[cols].isna().sum())
print("PHYSICALID with commas:", cs["PHYSICALID"].astype(str).str.contains(",", regex=False).sum())

  PHYSICALID OBJECTID  JOINID  BPHYS_ID
0     46,810   42,563     NaN       NaN
1     86,757   78,537     NaN       NaN
2     84,282   76,298     NaN       NaN
3     79,741   72,173     NaN       NaN
4    184,200  115,877     NaN       NaN
5    170,796  110,562     NaN       NaN
6    108,922   98,416     NaN       NaN
7     31,380   28,373     NaN       NaN
PHYSICALID     object
OBJECTID       object
JOINID        float64
BPHYS_ID      float64
dtype: object
PHYSICALID         0
OBJECTID           0
JOINID        122311
BPHYS_ID      122311
dtype: int64
PHYSICALID with commas: 121511


In [36]:
def to_int(s):
    return pd.to_numeric(s.astype(str).str.replace(",", "", regex=False).str.strip(),
                         errors="coerce")

hs, as_ = set(hist["SegmentID"]), set(a["SegmentID"])
for c in cols:
    s = set(to_int(cs[c]).dropna().astype("int64"))
    print(f"{c:11s} unique={len(s):7d} | historical match={len(hs & s):5d} ({len(hs & s)/len(hs):.1%})"
          f" | automated match={len(as_ & s):5d} ({len(as_ & s)/len(as_):.1%})")

PHYSICALID  unique= 122274 | historical match= 1055 (53.9%) | automated match= 2118 (62.5%)
OBJECTID    unique= 122311 | historical match= 1098 (56.1%) | automated match= 2168 (63.9%)
JOINID      unique=      0 | historical match=    0 (0.0%) | automated match=    0 (0.0%)
BPHYS_ID    unique=      0 | historical match=    0 (0.0%) | automated match=    0 (0.0%)


In [37]:
ID_COL = "PHYSICALID"

cs["_id"] = to_int(cs[ID_COL])
cs1 = cs.dropna(subset=["_id"]).copy()
cs1["_id"] = cs1["_id"].astype("int64")
print("Centerline rows with a numeric id:", len(cs1), "| duplicate ids:", cs1["_id"].duplicated().sum())

cs_name = norm(cs1.drop_duplicates("_id").set_index("_id")["Full Street Name"])

hn_all = hist.assign(n=norm(hist["Roadway Name"])).groupby("SegmentID")["n"].agg(set)
an_all = a.assign(n=norm(a["street"])).groupby("SegmentID")["n"].agg(set)

def agree(names_by_seg):
    out = {}
    for sid, names in names_by_seg.items():
        if sid in cs_name.index:
            c = cs_name[sid]
            out[sid] = any((c in n) or (n in c) for n in names)
    return pd.Series(out)

for label, nb in [("historical", hn_all), ("automated", an_all)]:
    r = agree(nb)
    print(f"{label}: in centerline {len(r)} of {len(nb)} | name agrees {int(r.sum())} ({r.mean():.1%})")

Centerline rows with a numeric id: 122311 | duplicate ids: 37
historical: in centerline 1055 of 1956 | name agrees 1 (0.1%)
automated: in centerline 2118 of 3391 | name agrees 5 (0.2%)


In [38]:
# Fractional counts in historical
nonint = hist[hour_cols].notna() & (hist[hour_cols] % 1 != 0)
print("Non-integer hourly cells:", int(nonint.sum().sum()))
print(hist.loc[nonint.any(axis=1), ["SegmentID", "Direction", "Date"]].head())

# Month coverage (autumn concentration?)
print(hist["Date"].dt.month.value_counts().sort_index())
print(a["ts"].dt.month.value_counts().sort_index())

# Large SegmentIDs (different scheme?)
big = hist["SegmentID"] > 300000
print("Historical rows with SegmentID > 300000:", int(big.sum()))
print(sorted(hist.loc[big, "SegmentID"].unique())[:15])

Non-integer hourly cells: 8
      SegmentID Direction       Date
6882      70873        NB 2012-09-30
6883      70873        NB 2012-10-01
Date
1      1881
2      3017
3       842
4       669
5       486
9      6180
10    20261
11     9403
12       17
Name: count, dtype: int64
ts
1     142098
2     137726
3     189923
4     176408
5     193207
6     186330
7     117914
8     132955
9     147317
10    192918
11    161860
12     96498
Name: count, dtype: int64
Historical rows with SegmentID > 300000: 2000
[np.int64(800251), np.int64(1553304), np.int64(8102302), np.int64(8103575), np.int64(9000022), np.int64(9000174), np.int64(9000265), np.int64(9001099), np.int64(9001153), np.int64(9001278), np.int64(9001279), np.int64(9001489), np.int64(9001519), np.int64(9001818), np.int64(9001819)]


In [40]:
# pip install geopandas pyogrio
import geopandas as gpd, pyogrio

LION_PATH = r"C:/Users/anasz/Desktop/uni bitch/FYP shi/data/lion/lion.gdb"          # .gdb folder or .shp file
if LION_PATH.lower().endswith(".gdb"):
    print(pyogrio.list_layers(LION_PATH))   # tell me which layers exist
    lion = gpd.read_file(LION_PATH, layer="lion")
else:
    lion = gpd.read_file(LION_PATH)

print(lion.shape, lion.crs)
print(lion.columns.tolist())

[['node' 'Point']
 ['node_stname' None]
 ['altnames' None]
 ['lion' 'MultiLineString']]
(243237, 129) EPSG:2263
['Street', 'SAFStreetName', 'FeatureTyp', 'SegmentTyp', 'IncExFlag', 'RB_Layer', 'NonPed', 'TrafDir', 'TrafSrc', 'SpecAddr', 'FaceCode', 'SeqNum', 'StreetCode', 'SAFStreetCode', 'LGC1', 'LGC2', 'LGC3', 'LGC4', 'LGC5', 'LGC6', 'LGC7', 'LGC8', 'LGC9', 'BOE_LGC', 'SegmentID', 'SegCount', 'LocStatus', 'LZip', 'RZip', 'LBoro', 'RBoro', 'L_CD', 'R_CD', 'LATOMICPOLYGON', 'RATOMICPOLYGON', 'LCT2020', 'LCT2020Suf', 'RCT2020', 'RCT2020Suf', 'LCB2020', 'LCB2020Suf', 'RCB2020', 'RCB2020Suf', 'LCT2010', 'LCT2010Suf', 'RCT2010', 'RCT2010Suf', 'LCB2010', 'LCB2010Suf', 'RCB2010', 'RCB2010Suf', 'LCT2000', 'LCT2000Suf', 'RCT2000', 'RCT2000Suf', 'LCB2000', 'LCB2000Suf', 'RCB2000', 'RCB2000Suf', 'LCT1990', 'LCT1990Suf', 'RCT1990', 'RCT1990Suf', 'LAssmDist', 'LElectDist', 'RAssmDist', 'RElectDist', 'SplitElect', 'LSchlDist', 'RSchlDist', 'SplitSchl', 'LSubSect', 'RSubSect', 'SanDistInd', 'MapFrom

In [41]:
NAME_COL = "Street"      # change if the printed columns use another street-name field

def to_int(s):
    return pd.to_numeric(s.astype(str).str.replace(",", "", regex=False).str.strip(), errors="coerce")

hs, as_ = set(hist["SegmentID"]), set(a["SegmentID"])
lion_name = norm(lion[NAME_COL])

for c in lion.columns:
    if c in ("geometry", NAME_COL):
        continue
    v = to_int(lion[c])
    if v.notna().mean() < 0.5 or v.nunique() < 50000:
        continue
    ids = v.astype("Int64")
    s = set(ids.dropna().astype("int64"))
    # name agreement on matched rows (first row per id)
    nm = pd.Series(lion_name.values, index=ids).dropna()
    nm = nm[~nm.index.duplicated()]
    hn_ = hist.assign(n=norm(hist["Roadway Name"])).groupby("SegmentID")["n"].agg(set)
    hit = [any((nm[sid] in n) or (n in nm[sid]) for n in names)
           for sid, names in hn_.items() if sid in nm.index]
    print(f"{c:20s} hist match={len(hs & s)/len(hs):.1%} automated match={len(as_ & s)/len(as_):.1%}"
          f" | hist name agrees={np.mean(hit):.1%} on {len(hit)} matched")

SegmentID            hist match=87.7% automated match=85.3% | hist name agrees=93.9% on 1716 matched


C:\ProgramData\anaconda3\Lib\site-packages\numpy\_core\fromnumeric.py:3904: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
C:\ProgramData\anaconda3\Lib\site-packages\numpy\_core\_methods.py:147: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


XFrom                hist match=0.0% automated match=0.0% | hist name agrees=nan% on 0 matched
YFrom                hist match=18.6% automated match=17.5% | hist name agrees=0.0% on 363 matched


C:\ProgramData\anaconda3\Lib\site-packages\numpy\_core\fromnumeric.py:3904: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
C:\ProgramData\anaconda3\Lib\site-packages\numpy\_core\_methods.py:147: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


XTo                  hist match=0.0% automated match=0.0% | hist name agrees=nan% on 0 matched
YTo                  hist match=18.8% automated match=17.6% | hist name agrees=0.0% on 367 matched
NodeIDFrom           hist match=52.9% automated match=57.4% | hist name agrees=0.5% on 1034 matched
NodeIDTo             hist match=53.4% automated match=57.9% | hist name agrees=0.4% on 1044 matched
PhysicalID           hist match=54.2% automated match=62.5% | hist name agrees=0.1% on 1061 matched
GenericID            hist match=54.8% automated match=62.2% | hist name agrees=0.0% on 1071 matched


C:\ProgramData\anaconda3\Lib\site-packages\numpy\_core\fromnumeric.py:3904: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
C:\ProgramData\anaconda3\Lib\site-packages\numpy\_core\_methods.py:147: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


LBlockFaceID         hist match=0.0% automated match=0.0% | hist name agrees=nan% on 0 matched


C:\ProgramData\anaconda3\Lib\site-packages\numpy\_core\fromnumeric.py:3904: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
C:\ProgramData\anaconda3\Lib\site-packages\numpy\_core\_methods.py:147: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


RBlockFaceID         hist match=0.0% automated match=0.0% | hist name agrees=nan% on 0 matched
LegacyID             hist match=88.1% automated match=93.4% | hist name agrees=93.7% on 1724 matched


TypeError: cannot safely cast non-equivalent object to int64

In [42]:
def to_int_series(s):
    return pd.to_numeric(s.astype(str).str.replace(",", "", regex=False).str.strip(), errors="coerce")

def id_map(col):
    d = lion[[col, "Street"]].copy()
    d["id"] = to_int_series(d[col])
    d = d.dropna(subset=["id"])
    d = d[d["id"] % 1 == 0]
    d["id"] = d["id"].astype("int64")
    d["nm"] = norm(d["Street"])
    return d.drop_duplicates("id").set_index("id")["nm"]

by_seg, by_leg = id_map("SegmentID"), id_map("LegacyID")

def check(names_by_seg, label):
    rows = []
    for sid, names in names_by_seg.items():
        r = {"SegmentID": sid}
        for tag, m in [("seg", by_seg), ("leg", by_leg)]:
            hit = sid in m.index
            r[tag] = hit
            r[tag + "_ok"] = hit and any((m[sid] in n) or (n in m[sid]) for n in names)
        rows.append(r)
    t = pd.DataFrame(rows).set_index("SegmentID")
    t["any"], t["any_ok"] = t["seg"] | t["leg"], t["seg_ok"] | t["leg_ok"]
    print(f"{label}: {len(t)} segments")
    for tag, nm in [("seg", "SegmentID"), ("leg", "LegacyID"), ("any", "either")]:
        print(f"  {nm:9s} matched {t[tag].sum():5d} ({t[tag].mean():.1%}) | names agree {t[tag + '_ok'].sum()} of {t[tag].sum()}")
    print("  neither:", (~t["any"]).sum())
    return t

th = check(hn_all, "historical")
ta = check(an_all, "automated")

historical: 1956 segments
  SegmentID matched  1716 (87.7%) | names agree 1612 of 1716
  LegacyID  matched  1724 (88.1%) | names agree 1615 of 1724
  either    matched  1920 (98.2%) | names agree 1800 of 1920
  neither: 36
automated: 3391 segments
  SegmentID matched  2893 (85.3%) | names agree 2441 of 2893
  LegacyID  matched  3167 (93.4%) | names agree 2613 of 3167
  either    matched  3317 (97.8%) | names agree 2751 of 3317
  neither: 74


In [43]:
# Is the name mismatch concentrated in early (2012) segments?
fy = hist.groupby("SegmentID")["Date"].min().dt.year.rename("first_year")
tt = th.join(fy)
print(tt[tt["seg"]].groupby("first_year")["seg_ok"].agg(["mean", "size"]).round(3))

# Examples of matched-but-disagreeing names
for sid in th[th["seg"] & ~th["seg_ok"]].index[:15]:
    print(sid, "| hist:", list(hn_all[sid])[:2], "| LION:", by_seg[sid])

# Unmatched historical segments and the 7-digit IDs
un = th[~th["any"]].index
sub = hist[hist["SegmentID"].isin(un)]
print("\nUnmatched historical segments:", len(un), "| rows:", len(sub))
print(sub.groupby(sub["Date"].dt.year).size())w
big_ids = hist.loc[hist["SegmentID"] > 300000, ["SegmentID", "Roadway Name"]].drop_duplicates()
print(big_ids.head(20))
print("Share of large IDs found in LION:", round(big_ids["SegmentID"].isin(by_seg.index).mean(), 3))

             mean  size
first_year             
2012        0.954   634
2013        1.000     9
2014        0.908   400
2015        0.948   252
2016        0.934   136
2017        0.929   127
2018        1.000    29
2019        0.962    80
2020        0.905    42
2021        1.000     7
202 | hist: ['MAIN STREET'] | LION: GIRARD STREET
1884 | hist: ['FOSTER ROAD'] | LION: LAMONT AVENUE
1886 | hist: ['FOSTER ROAD'] | LION: IONIA AVENUE
18562 | hist: ['SHORE PKWY'] | LION: SHORE PARKWAY
19555 | hist: ['FT HAMILTON PARKWAY'] | LION: FORT HAMILTON PARKWAY
19559 | hist: ['FT HAMILTON PARKWAY'] | LION: FORT HAMILTON PARKWAY
21018 | hist: ['3RD AVENUE'] | LION: 3 AVENUE
21022 | hist: ['3RD AVENUE'] | LION: 3 AVENUE
23942 | hist: ['MARKET STREET'] | LION: MARKET SLIP
26075 | hist: ['SHORE PKWY SOUTH'] | LION: SHORE PARKWAY
33798 | hist: ['6TH AVENUE', '6 AVENUE'] | LION: AVENUE OF THE AMERICAS
34669 | hist: ['FDR DRIVE'] | LION: FRANKLIN D ROOSEVELT DRIVE
38837 | hist: ['MT MORRIS PARK WEST'] 

In [44]:
lg = lion[["SegmentID", "geometry"]].copy()
lg["sid"] = to_int_series(lg["SegmentID"])
lg = lg.dropna(subset=["sid"])
lg = lg[lg["sid"] % 1 == 0].drop_duplicates("sid")
lg["sid"] = lg["sid"].astype("int64")
lg = lg.set_index("sid")["geometry"]

ap = a.drop_duplicates("SegmentID")[["SegmentID", "WktGeom"]]
ap = ap[ap["SegmentID"].isin(lg.index)]
pts = gpd.GeoSeries.from_wkt(ap["WktGeom"].values, crs="EPSG:2263")
lines = gpd.GeoSeries(lg.loc[ap["SegmentID"]].values, crs="EPSG:2263")

dist = pts.distance(lines)   # feet
print("Segments checked:", len(dist))
print(dist.describe(percentiles=[.5, .75, .9, .95, .99]))
print("Within 100 ft:", round((dist <= 100).mean(), 3))

Segments checked: 2893
count    2.893000e+03
mean     5.330597e-01
std      3.391315e+00
min      6.130817e-09
50%      7.171245e-03
75%      3.333113e-02
90%      2.244870e-01
95%      2.601624e+00
99%      1.188419e+01
max      1.273207e+02
dtype: float64
Within 100 ft: 1.0


In [45]:
a = a.sort_values(["SegmentID", "Direction", "ts"]).reset_index(drop=True)
s = a["Vol"].eq(0)
grp = (a["SegmentID"] != a["SegmentID"].shift()) | (a["Direction"] != a["Direction"].shift())
run_id = (s != s.shift()).cumsum() + grp.cumsum() * 0
run_len = s.groupby(run_id).transform("size")
zero_run = np.where(s, run_len, 0)

for thr in [1, 2, 4, 8, 16, 32]:
    n = int(((zero_run >= thr) & s).sum())
    print(f"zero runs >= {thr:2d} intervals: {n:8d} cells ({n / s.sum():.1%} of zeros)")

zero runs >=  1 intervals:    96278 cells (100.0% of zeros)
zero runs >=  2 intervals:    86461 cells (89.8% of zeros)
zero runs >=  4 intervals:    77660 cells (80.7% of zeros)
zero runs >=  8 intervals:    69850 cells (72.6% of zeros)
zero runs >= 16 intervals:    62964 cells (65.4% of zeros)
zero runs >= 32 intervals:    56146 cells (58.3% of zeros)


In [46]:
la = lion.assign(sid=to_int_series(lion["SegmentID"]), lid=to_int_series(lion["LegacyID"]))
both = la.dropna(subset=["sid", "lid"]).drop_duplicates("sid")
conflict = both[both["lid"] != both["sid"]]
print("LION rows where SegmentID != LegacyID:", len(conflict))

LION rows where SegmentID != LegacyID: 82070


# Automated Cleaning

## Block A: load, convert, quarantine

In [47]:
import numpy as np, pandas as pd

AUTO_PATH = r"C:/Users/anasz/Desktop/uni bitch/FYP shi/data/Automated_Traffic_Volume_Counts_20260928.csv"
raw = pd.read_csv(AUTO_PATH)
print("raw:", raw.shape)

d = raw.copy()
d["Vol"] = pd.to_numeric(d["Vol"].astype(str).str.replace(",", "", regex=False).str.strip(), errors="coerce")
print("Vol not numeric:", int(d["Vol"].isna().sum()))
neg = d["Vol"] < 0
print("Negative Vol set to missing:", int(neg.sum()))
d.loc[neg, "Vol"] = np.nan

d["ts"] = pd.to_datetime(dict(year=d["Yr"], month=d["M"], day=d["D"], hour=d["HH"], minute=d["MM"]), errors="coerce")
print("Unparsed timestamps:", int(d["ts"].isna().sum()))
d["date"] = d["ts"].dt.normalize()
d["hour_start"] = d["ts"].dt.floor("h")

# Quarantine, in priority order so each row gets one reason
d["q_reason"] = pd.Series(pd.NA, index=d.index, dtype="object")
d.loc[d["MM"].isin([10, 20, 40, 50]), "q_reason"] = "odd_minute"
d.loc[d["q_reason"].isna() & d["Direction"].isin(["EW", "NS"]), "q_reason"] = "two_way"
rem = d["q_reason"].isna()
dup = d.loc[rem].duplicated(["SegmentID", "Direction", "ts"], keep=False)
d.loc[dup[dup].index, "q_reason"] = "duplicate_key"
print(d["q_reason"].value_counts(dropna=False))

# Study type per RequestID, segment, direction (ignoring quarantined rows)
study = ["RequestID", "SegmentID", "Direction"]
mins = d[d["q_reason"].isna()].groupby(study)["MM"].agg(lambda s: frozenset(s.unique()))
stype = mins.map(lambda m: "hourly" if m <= {0} else "15min").rename("study_type")
d = d.join(stype, on=study)
d["interval_min"] = np.where(d["study_type"] == "hourly", 60, 15)
print(d.loc[d["q_reason"].isna(), "study_type"].value_counts())

raw: (1875154, 14)
Vol not numeric: 0
Negative Vol set to missing: 1
Unparsed timestamps: 0
q_reason
<NA>             1855463
duplicate_key      10292
two_way             6528
odd_minute          2871
Name: count, dtype: int64
study_type
15min     1844480
hourly      10983
Name: count, dtype: int64


## Block B: zero rule and hourly build

In [48]:
ALWAYS_MIN, EXPECTED_MIN = 120, 8

clean = d[d["q_reason"].isna()].sort_values(study + ["ts"]).reset_index(drop=True)
clean["hr"] = clean["ts"].dt.hour

typ = (clean[clean["Vol"] > 0]
       .groupby(["SegmentID", "Direction", "interval_min", "hr"])["Vol"].median()
       .rename("typ").reset_index())
clean = clean.merge(typ, on=["SegmentID", "Direction", "interval_min", "hr"], how="left")

is0 = clean["Vol"].eq(0)
step = pd.to_timedelta(clean["interval_min"], unit="min")
new_run = ((is0 != is0.shift())
           | (clean[study] != clean[study].shift()).any(axis=1)
           | (clean["ts"].diff() > step))
run_len = is0.groupby(new_run.cumsum()).transform("size").where(is0, 0)
run_min = run_len * clean["interval_min"]
expected = run_len * clean["typ"].fillna(0)

outage = is0 & ((run_min >= ALWAYS_MIN) | (expected >= EXPECTED_MIN))
print("Zero cells:", int(is0.sum()))
print("  missing, run >= 2 h:", int((is0 & (run_min >= ALWAYS_MIN)).sum()))
print("  missing, shorter run but expected count >= 8:", int((outage & (run_min < ALWAYS_MIN)).sum()))
print("  kept as real zeros:", int((is0 & ~outage).sum()))
clean["zero_to_missing"] = outage
clean.loc[outage, "Vol"] = np.nan

# Hourly rows: 15-min studies need all 4 intervals, hourly studies need 1 value
g = clean.groupby(["RequestID", "SegmentID", "Direction", "hour_start", "interval_min"], sort=False)
hourly = g["Vol"].agg(vol_sum="sum", n_valid="count").reset_index()
need = np.where(hourly["interval_min"] == 60, 1, 4)
hourly["volume"] = np.where(hourly["n_valid"] == need, hourly["vol_sum"], np.nan)
print("Hour cells built:", len(hourly), "| valid:", int(hourly["volume"].notna().sum()),
      "| missing (incomplete or outage):", int(hourly["volume"].isna().sum()))

ok = hourly[hourly["volume"].notna()].copy()
col = ok.duplicated(["SegmentID", "Direction", "hour_start"], keep=False)
print("Hour collisions between different requests:", int(col.sum()))
hour_collision = ok[col]
ok = ok[~col].copy()

ok["date"] = ok["hour_start"].dt.normalize()
ok["hours_in_day"] = ok.groupby(["SegmentID", "Direction", "date"])["volume"].transform("size")
ok["partial_day"] = ok["hours_in_day"] < 24
ok["covid"] = ok["hour_start"] >= "2020-03-01"
ok["sparse_year"] = ok["hour_start"].dt.year.isin([2000, 2006, 2007, 2008, 2026])
tz = ok["hour_start"].dt.tz_localize("America/New_York", ambiguous="NaT", nonexistent="NaT")
ok["hour_start_tz"] = tz
ok["dst_issue"] = tz.isna()
print("DST-affected hours:", int(ok["dst_issue"].sum()), "| partial-day hours:", int(ok["partial_day"].sum()))

Zero cells: 95278
  missing, run >= 2 h: 67965
  missing, shorter run but expected count >= 8: 9822
  kept as real zeros: 17491
Hour cells built: 475854 | valid: 444140 | missing (incomplete or outage): 31714
Hour collisions between different requests: 0
DST-affected hours: 105 | partial-day hours: 81284


## Block C: LION join with spatial validation

In [49]:
import geopandas as gpd
from pyproj import Transformer

def id_to_row(col_name):
    v = to_int_series(lion[col_name])
    m = (v.notna() & (v % 1 == 0)).values
    s = pd.Series(lion.index[m], index=v[m].astype("int64").values)
    return s[~s.index.duplicated()]

seg_row, leg_row = id_to_row("SegmentID"), id_to_row("LegacyID")

segs = d.drop_duplicates("SegmentID")[["SegmentID", "Boro", "WktGeom"]].reset_index(drop=True)
pts = gpd.GeoSeries.from_wkt(segs["WktGeom"], crs="EPSG:2263")

def cand_dist(row_map):
    rows = segs["SegmentID"].map(row_map)
    has = rows.notna()
    dist = pd.Series(np.nan, index=segs.index)
    if has.any():
        lines = gpd.GeoSeries(lion.geometry.loc[rows[has].astype(int).values].values,
                              index=segs.index[has], crs="EPSG:2263")
        dist[has] = pts[has].distance(lines)
    return rows, dist

segs["seg_row"], segs["seg_dist"] = cand_dist(seg_row)
segs["leg_row"], segs["leg_dist"] = cand_dist(leg_row)

MAXD = 100
seg_ok, leg_ok = segs["seg_dist"] <= MAXD, segs["leg_dist"] <= MAXD
use_seg = seg_ok & (~leg_ok | (segs["seg_dist"] <= segs["leg_dist"]))
use_leg = leg_ok & ~use_seg
segs["lion_key_used"] = np.select([use_seg, use_leg], ["SegmentID", "LegacyID"], default="none")
segs["lion_row"] = np.where(use_seg, segs["seg_row"], np.where(use_leg, segs["leg_row"], np.nan))
segs["key_conflict"] = segs["seg_row"].notna() & segs["leg_row"].notna() & (segs["seg_row"] != segs["leg_row"])

print(segs["lion_key_used"].value_counts())
print("Matched by both keys to DIFFERENT LION rows:", int(segs["key_conflict"].sum()))
print(segs.loc[segs["lion_key_used"] != "none", ["seg_dist", "leg_dist"]].describe())

attrs = [c for c in ["POSTED_SPEED", "Number_Travel_Lanes", "Number_Park_Lanes", "Number_Total_Lanes",
                     "StreetWidth_Min", "StreetWidth_Max", "StreetWidth_Irr", "FCC", "TrafDir",
                     "TRUCK_ROUTE_TYPE", "Snow_Priority", "BikeLane", "LBoro", "RBoro", "SHAPE_Length"]
         if c in lion.columns]
m = segs["lion_row"].notna()
la = lion.loc[segs.loc[m, "lion_row"].astype(int).values, attrs + ["geometry"]].copy()
la.index = segs.index[m]
rep = la.geometry.representative_point()
la["lon"], la["lat"] = Transformer.from_crs("EPSG:2263", "EPSG:4326", always_xy=True).transform(rep.x.values, rep.y.values)
la["lion_wkt_2263"] = la.geometry.to_wkt()
seg_table = segs[["SegmentID", "Boro", "lion_key_used", "key_conflict", "seg_dist", "leg_dist"]].join(la.drop(columns="geometry"))
print(seg_table.shape)

lion_key_used
SegmentID    2892
LegacyID      335
none          164
Name: count, dtype: int64
Matched by both keys to DIFFERENT LION rows: 0
           seg_dist      leg_dist
count  2.892000e+03  3.077000e+03
mean   4.872560e-01  2.518490e+00
std    2.442399e+00  1.129696e+01
min    6.130817e-09  6.130817e-09
25%    6.875643e-05  7.839194e-05
50%    5.485202e-03  1.089693e-02
75%    3.259520e-02  4.010125e-02
max    4.819623e+01  9.979461e+01
(3391, 24)


## Block D: assemble, reconcile, save

In [50]:
auto_hourly = ok.merge(seg_table, on="SegmentID", how="left")
auto_hourly = auto_hourly.rename(columns={"SegmentID": "segment_id", "Direction": "direction",
                                          "interval_min": "orig_interval_min", "RequestID": "request_id"})
auto_hourly["source"] = "automated"

n_q = int(d["q_reason"].notna().sum())
print("raw rows:", len(raw), "| quarantined:", n_q, "| entering cleaning:", len(clean))
assert len(raw) == n_q + len(clean), "reconciliation failed"
print("final hourly rows:", len(auto_hourly), "| segments:", auto_hourly["segment_id"].nunique())
print(auto_hourly["lion_key_used"].value_counts(dropna=False))

import os
OUT = r"C:/Users/anasz/Desktop/uni bitch/FYP shi/data/clean"
os.makedirs(OUT, exist_ok=True)
try:
    auto_hourly.to_parquet(f"{OUT}/auto_hourly.parquet", index=False)
except Exception as e:
    print("parquet failed, writing csv instead:", e)
    auto_hourly.to_csv(f"{OUT}/auto_hourly.csv.gz", index=False)
d[d["q_reason"].notna()].to_csv(f"{OUT}/auto_quarantine.csv.gz", index=False)

raw rows: 1875154 | quarantined: 19691 | entering cleaning: 1855463
final hourly rows: 444140 | segments: 2843
lion_key_used
SegmentID    368937
LegacyID      53329
none          21874
Name: count, dtype: int64


## Block E: diagnostics

In [51]:
# 1. Why do some segments have no valid hourly rows?
all_seg = set(d["SegmentID"]); kept_seg = set(auto_hourly["segment_id"])
lost = sorted(all_seg - kept_seg)
print("Segments in raw:", len(all_seg), "| with valid hourly rows:", len(kept_seg), "| without:", len(lost))

L = d[d["SegmentID"].isin(lost)]
print("Rows in those segments:", len(L))
print(L["q_reason"].fillna("clean").value_counts())

cl = clean[clean["SegmentID"].isin(lost)]
print("Clean rows:", len(cl), "| zero_to_missing rows:", int(cl["zero_to_missing"].sum()))
hl = hourly[hourly["SegmentID"].isin(lost)]
print("Hour cells:", len(hl))
print("Valid intervals per hour cell:\n", hl["n_valid"].value_counts().sort_index())

seg_cat = pd.Series("no_complete_hour", index=lost)
has_clean = clean.groupby("SegmentID").size()
seg_cat[[s for s in lost if s not in has_clean.index]] = "all_rows_quarantined"
allz = cl.groupby("SegmentID")["zero_to_missing"].agg(["sum", "size"])
seg_cat[seg_cat.index.isin(allz[allz["sum"] == allz["size"]].index)] = "all_rows_outage_zeros"
print(seg_cat.value_counts())

Segments in raw: 3391 | with valid hourly rows: 2843 | without: 548
Rows in those segments: 18950
q_reason
clean      13370
two_way     5580
Name: count, dtype: int64
Clean rows: 13370 | zero_to_missing rows: 6308
Hour cells: 4562
Valid intervals per hour cell:
 n_valid
0    1608
1     413
2     974
3    1567
Name: count, dtype: int64
no_complete_hour         531
all_rows_outage_zeros     14
all_rows_quarantined       3
Name: count, dtype: int64


In [52]:
# 2. Where do the kept zeros fall?
kept0 = clean[clean["Vol"] == 0]
print("Real zeros kept:", len(kept0))
print(kept0["hr"].value_counts().sort_index())

# 3. RequestID 7620 (FDR Drive 2007): is it classified as hourly?
print(clean.loc[clean["RequestID"] == 7620, ["ts", "Vol", "study_type"]].head(10))
hs = auto_hourly[auto_hourly["orig_interval_min"] == 60]
print("Hourly-study daily totals:")
print(hs.groupby(["segment_id", "direction", "date"])["volume"].sum().describe())

# 4. Final table sanity
print(auto_hourly["volume"].describe(percentiles=[.01, .05, .5, .95, .99, .999]))
print("Zero hours:", int((auto_hourly["volume"] == 0).sum()))
print(auto_hourly.groupby(auto_hourly["hour_start"].dt.hour)["volume"].median().round(0))
print(auto_hourly.groupby(auto_hourly["hour_start"].dt.year).size())

Real zeros kept: 17491
hr
0     1114
1     1689
2     2535
3     3032
4     2450
5     1315
6      633
7      334
8      258
9      212
10     233
11     204
12     183
13     161
14     158
15     160
16     197
17     158
18     186
19     222
20     306
21     425
22     555
23     771
Name: count, dtype: int64
                        ts     Vol study_type
231325 2007-10-20 08:00:00  3745.0     hourly
231326 2007-10-20 09:00:00  3741.0     hourly
231327 2007-10-20 10:00:00  3903.0     hourly
231328 2007-10-20 11:00:00  4057.0     hourly
231329 2007-10-20 12:00:00  4084.0     hourly
231330 2007-10-20 13:00:00  4433.0     hourly
231331 2007-10-20 14:00:00  4567.0     hourly
231332 2007-10-20 15:00:00  4642.0     hourly
231333 2007-10-20 16:00:00  4584.0     hourly
231334 2007-10-20 17:00:00  4593.0     hourly
Hourly-study daily totals:
count      597.000000
mean     14970.437186
std      20685.357190
min          0.000000
25%       1398.000000
50%       6512.000000
75%      16051.0000

In [53]:
none = segs[segs["lion_key_used"] == "none"]
npts = gpd.GeoDataFrame({"SegmentID": none["SegmentID"].values},
                        geometry=pts[none.index].values, crs="EPSG:2263")
cand = lion[["Street", "geometry"]].copy()
cand["lion_row"] = cand.index

near = gpd.sjoin_nearest(npts, cand, how="left", distance_col="d_ft")
near = near.sort_values("d_ft").drop_duplicates("SegmentID")
print("Segments checked:", len(near))
print(near["d_ft"].describe(percentiles=[.5, .75, .9, .95]))

nm = (d[d["SegmentID"].isin(near["SegmentID"])]
      .assign(n=lambda x: norm(x["street"])).groupby("SegmentID")["n"].agg(set))
near["lion_nm"] = norm(near["Street"])
near["name_ok"] = [any((l in n) or (n in l) for n in nm[s])
                   for s, l in zip(near["SegmentID"], near["lion_nm"])]
print("Nearest line name agrees:", int(near["name_ok"].sum()), "of", len(near))
print(near.groupby(near["d_ft"] <= 30)["name_ok"].agg(["mean", "size"]))

Segments checked: 164
count    1.640000e+02
mean     3.653010e+00
std      7.807096e+00
min      7.570842e-07
50%      4.118080e-02
75%      2.876990e+00
90%      1.284363e+01
95%      2.038313e+01
max      4.736462e+01
Name: d_ft, dtype: float64
Nearest line name agrees: 96 of 164
         mean  size
d_ft               
False  0.5000     4
True   0.5875   160


In [54]:
L.to_csv(f"{OUT}/auto_fragments.csv.gz", index=False)

In [56]:
pick = near.copy()
pick["final_key"] = np.where(pick["name_ok"], "spatial", "none")

# Sample of the disagreeing matches, to judge whether the name check is too strict
bad = pick[~pick["name_ok"]].sample(20, random_state=1)[["SegmentID", "Street", "d_ft"]]
bad["recorded_names"] = bad["SegmentID"].map(lambda s: sorted(nm.get(s, [])))
print(bad.to_string())
print("Adopted:", int((pick["final_key"] == "spatial").sum()),
      "| left unmatched:", int((pick["final_key"] == "none").sum()))

     SegmentID                            Street       d_ft                                                                recorded_names
140     154271            SHORE PARKWAY GREENWAY  18.815534                                             [W B MILL BASIN BRIDGE BELT PKWY]
29       21729                    DELAVAN STREET   1.435533                                                              [DELEVAN STREET]
135     118085              NORTH CONDUIT AVENUE   7.298076                                   [W B N CONDUIT AVENUE BROOKVILLE BOULEVARD]
131     132886   145 ST BRIDGE PED AND BIKE PATH  10.487409                                                             [W 145 STREET BR]
80      184144     VAN WYCK EXPRESSWAY NB EXIT 9  25.279056                                               [VAN WYCK EXPRESSWAY EXIT 9 NB]
19      144320     PEDESTRIAN AND BIKE PATH LINK  13.479127                                                           [KOSCIUSZKO BRIDGE]
18      146439          LGA TERMIN

In [57]:
def norm2(s):
    s = s.astype(str).str.upper().str.replace(r"[^A-Z0-9 ]", " ", regex=True)
    subs = [(r"\bE\b", "EAST"), (r"\bW\b", "WEST"), (r"\bN\b", "NORTH"), (r"\bS\b", "SOUTH"),
            (r"\bAV\b|\bAVE\b", "AVENUE"), (r"\bST\b", "STREET"), (r"\bDR\b", "DRIVE"),
            (r"\bTER\b", "TERRACE"), (r"\bRD\b", "ROAD"), (r"\bBLVD\b", "BOULEVARD"),
            (r"\bPKWY\b", "PARKWAY"), (r"\bBR\b|\bBRDG\b", "BRIDGE"), (r"\bFT\b", "FORT"),
            (r"\bMT\b", "MOUNT"), (r"\bGDNS\b", "GARDENS")]
    for pat, rep in subs:
        s = s.str.replace(pat, rep, regex=True)
    return s.str.replace(r"\s+", " ", regex=True).str.strip()

def name_match(rec, lion_name):
    # Any recorded name that contains or is contained in the LION name, after normalizing both
    L = norm2(pd.Series([lion_name])).iloc[0]
    return any((L in r) or (r in L) for r in rec)

rec_names = d[d["SegmentID"].isin(near["SegmentID"])].assign(n=lambda x: norm2(x["street"]))\
            .groupby("SegmentID")["n"].agg(lambda s: set(s))
near["name_ok2"] = [name_match(rec_names.get(s, set()), l) for s, l in zip(near["SegmentID"], near["Street"])]
print("Name agrees (old):", int(near["name_ok"].sum()), "| (new):", int(near["name_ok2"].sum()), "of", len(near))
print(near.loc[near["name_ok2"] & ~near["name_ok"], ["SegmentID", "Street", "d_ft"]].head(20).to_string())

Name agrees (old): 96 | (new): 111 of 164
     SegmentID                Street       d_ft
26       69068       WEST 124 STREET   0.002330
149      33129        EAST 14 STREET   0.012690
62       34110        WEST 55 STREET   0.025991
10       82336              62 DRIVE   0.026494
151     115260         TERRACE DRIVE   0.028245
120     151043    VETERANS ROAD WEST   0.033083
105     191117         OCEAN PARKWAY   0.033612
133     164326        EAST 17 STREET   0.296088
139     118752        WEST 95 STREET   0.333593
54       38828       WEST 120 STREET   0.639580
66       88130    EAST GUN HILL ROAD   0.937163
1       132866   EAST TREMONT AVENUE   3.862488
135     118085  NORTH CONDUIT AVENUE   7.298076
38       37267       RIVERSIDE DRIVE   8.972536
130     132884     145 STREET BRIDGE  11.160902


In [59]:
# 1. Accepted matches from the improved name check
accept = near.loc[near["name_ok2"], ["SegmentID", "lion_row"]].copy()
accept["lion_row"] = accept["lion_row"].astype(int)
print("Accepted:", len(accept))

# 2. Update the segment table
segs2 = segs.copy()
rowmap = accept.set_index("SegmentID")["lion_row"]
hit = segs2["SegmentID"].isin(rowmap.index) & segs2["lion_row"].isna()
segs2.loc[hit, "lion_row"] = segs2.loc[hit, "SegmentID"].map(rowmap)
segs2.loc[hit, "lion_key_used"] = "spatial"
print(segs2["lion_key_used"].value_counts())

# 3. Rebuild LION attributes for every matched segment
has = segs2["lion_row"].notna()
la2 = lion.loc[segs2.loc[has, "lion_row"].astype(int).values, attrs + ["geometry"]].copy()
la2.index = segs2.index[has]
rep = la2.geometry.representative_point()
la2["lon"], la2["lat"] = Transformer.from_crs("EPSG:2263", "EPSG:4326", always_xy=True).transform(rep.x.values, rep.y.values)
la2["lion_wkt_2263"] = la2.geometry.to_wkt()
seg_table2 = segs2[["SegmentID", "Boro", "lion_key_used", "key_conflict", "seg_dist", "leg_dist"]].join(la2.drop(columns="geometry"))

# 4. Swap the LION columns in the hourly table (row count must not change)
new_cols = [c for c in seg_table2.columns if c != "SegmentID"]
auto_hourly2 = (auto_hourly.drop(columns=[c for c in new_cols if c in auto_hourly.columns])
                .merge(seg_table2.rename(columns={"SegmentID": "segment_id"}), on="segment_id", how="left"))

print("Rows before:", len(auto_hourly), "| after:", len(auto_hourly2))
assert len(auto_hourly2) == len(auto_hourly), "row count changed"
print(auto_hourly2["lion_key_used"].value_counts(dropna=False))
print("Rows by key:", auto_hourly2.groupby("lion_key_used").size().to_dict())

# 5. Save as a new version, keeping the old file
auto_hourly2.to_parquet(f"{OUT}/auto_hourly_v2.parquet", index=False)

Accepted: 111
lion_key_used
SegmentID    2892
LegacyID      335
spatial       111
none           53
Name: count, dtype: int64
Rows before: 444140 | after: 444140
lion_key_used
SegmentID    368937
LegacyID      53329
spatial       15526
none           6348
Name: count, dtype: int64
Rows by key: {'LegacyID': 53329, 'SegmentID': 368937, 'none': 6348, 'spatial': 15526}


In [60]:
chk = pd.read_parquet(f"{OUT}/auto_hourly_v2.parquet")
print("Reloaded rows:", len(chk), "| expected 444140")
print(chk["lion_key_used"].value_counts(dropna=False))

Reloaded rows: 444140 | expected 444140
lion_key_used
SegmentID    368937
LegacyID      53329
spatial       15526
none           6348
Name: count, dtype: int64


In [61]:
sp = (auto_hourly2[auto_hourly2["lion_key_used"] == "spatial"]
      .drop_duplicates("segment_id")[["segment_id", "Street", "POSTED_SPEED", "Number_Travel_Lanes", "seg_dist"]])
print(sp.sample(15, random_state=7).to_string())

KeyError: "['Street'] not in index"

In [62]:
    ids = sp.sample(15, random_state=7)["segment_id"]
rec = (d[d["SegmentID"].isin(ids)].drop_duplicates("SegmentID")
         .set_index("SegmentID")["street"])
print(rec.to_string())

NameError: name 'sp' is not defined

In [63]:
print(auto_hourly2.groupby("lion_key_used")[["POSTED_SPEED", "Number_Travel_Lanes"]]
      .agg(lambda s: s.notna().mean()).round(3))
print(auto_hourly2.loc[auto_hourly2["lion_key_used"] == "spatial", "POSTED_SPEED"].value_counts().head(10))

               POSTED_SPEED  Number_Travel_Lanes
lion_key_used                                   
LegacyID                1.0                  1.0
SegmentID               1.0                  1.0
none                    0.0                  0.0
spatial                 1.0                  1.0
POSTED_SPEED
25    10926
30     2169
       1749
35      383
50      216
40       83
Name: count, dtype: int64


In [64]:
sp_geo = auto_hourly2[auto_hourly2["lion_key_used"] == "spatial"].drop_duplicates("segment_id")
print(sp_geo.groupby("Boro")[["lat", "lon"]].median().round(3))

                  lat     lon
Boro                         
Bronx          40.835 -73.881
Brooklyn       40.664 -73.964
Manhattan      40.749 -73.984
Queens         40.742 -73.926
Staten Island  40.618 -74.170


In [65]:
# Step 2 fix: LION street name for each segment, via lion_row
seg_named = segs2[segs2["lion_row"].notna()].copy()
seg_named["lion_street"] = lion.loc[seg_named["lion_row"].astype(int).values, "Street"].values
lion_name_by_seg = seg_named.set_index("SegmentID")["lion_street"]

sp = (auto_hourly2[auto_hourly2["lion_key_used"] == "spatial"]
      .drop_duplicates("segment_id")[["segment_id", "Boro", "seg_dist"]]
      .assign(lion_street=lambda x: x["segment_id"].map(lion_name_by_seg)))
rec = d.drop_duplicates("SegmentID").set_index("SegmentID")["street"]
sp["recorded"] = sp["segment_id"].map(rec)

print(sp.sample(15, random_state=7).to_string())

        segment_id           Boro  seg_dist              lion_street                 recorded
86984       108151       Brooklyn       NaN          HAMILTON AVENUE          Hamilton Ave NB
405447     9008995      Manhattan       NaN             PEARL STREET             PEARL STREET
291262     9014434          Bronx       NaN      EAST TREMONT AVENUE      EAST TREMONT AVENUE
234524      109847       Brooklyn       NaN            FARRAGUT ROAD            FARRAGUT ROAD
153284       44129       Brooklyn       NaN  MARCUS GARVEY BOULEVARD  MARCUS GARVEY BOULEVARD
67007       141214          Bronx       NaN       EAST 174 ST BRIDGE          EAST 174 STREET
202984      266374       Brooklyn       NaN        VAN SICLEN AVENUE        VAN SICLEN AVENUE
149717      155167      Manhattan       NaN         MANHATTAN BRIDGE         MANHATTAN BRIDGE
92137         4466  Staten Island       NaN         CORTELYOU AVENUE         CORTELYOU AVENUE
144964      188147         Queens       NaN          SKILLMA

In [66]:
# Step 3 follow-up: what is the unlabelled POSTED_SPEED category?
sp_rows = auto_hourly2[auto_hourly2["lion_key_used"] == "spatial"]
print(sp_rows["POSTED_SPEED"].map(repr).value_counts().head(8))
print("Types:", sp_rows["POSTED_SPEED"].map(type).value_counts().to_dict())

POSTED_SPEED
'25'    10926
'30'     2169
'  '     1749
'35'      383
'50'      216
'40'       83
Name: count, dtype: int64
Types: {<class 'str'>: 15526}


In [67]:
# Step 4 (proper): does LION's borough agree with the recorded borough?
print(pd.crosstab(sp_rows.drop_duplicates("segment_id")["Boro"],
                  sp_rows.drop_duplicates("segment_id")["LBoro"]))

LBoro          1.0  2.0  3.0  4.0  5.0
Boro                                  
Bronx            0   21    0    0    0
Brooklyn         0    0   25    0    0
Manhattan       27    0    0    0    0
Queens           0    0    0   12    0
Staten Island    0    0    0    0    9


In [68]:
num_cols = [c for c in ["POSTED_SPEED", "Number_Travel_Lanes", "Number_Park_Lanes", "Number_Total_Lanes",
                        "StreetWidth_Min", "StreetWidth_Max", "StreetWidth_Irr", "SHAPE_Length",
                        "LBoro", "RBoro"] if c in auto_hourly2.columns]
cat_cols = [c for c in ["FCC", "TrafDir", "TRUCK_ROUTE_TYPE", "Snow_Priority", "BikeLane"]
            if c in auto_hourly2.columns]

av3 = auto_hourly2.copy()
for c in num_cols:
    s = av3[c].astype(str).str.strip().replace({"": np.nan, "nan": np.nan, "None": np.nan})
    av3[c] = pd.to_numeric(s, errors="coerce")
for c in cat_cols:
    s = av3[c].astype(str).str.strip()
    av3[c] = s.where(~s.isin(["", "nan", "None"]), np.nan)

assert len(av3) == 444140, "row count changed"

# Real fill rates by match type
print(av3.groupby("lion_key_used")[num_cols + cat_cols]
        .agg(lambda s: s.notna().mean()).round(3).T)
print(av3["POSTED_SPEED"].value_counts(dropna=False).head(8))

# The 17 spatial segments without a borough code
sp_seg = av3[av3["lion_key_used"] == "spatial"].drop_duplicates("segment_id")
miss = sp_seg[sp_seg["LBoro"].isna()]["segment_id"]
print("Spatial segments with no LBoro:", len(miss))
print(segs2[segs2["SegmentID"].isin(miss)]
      .assign(lion_street=lambda x: lion.loc[x["lion_row"].astype(int).values, "Street"].values)
      [["SegmentID", "Boro", "lion_street"]].head(20).to_string())

av3.to_parquet(f"{OUT}/auto_hourly_v3.parquet", index=False)

C:\Users\anasz\AppData\Local\Temp\ipykernel_6364\3907245332.py:9: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  s = av3[c].astype(str).str.strip().replace({"": np.nan, "nan": np.nan, "None": np.nan})


lion_key_used        LegacyID  SegmentID  none  spatial
POSTED_SPEED            0.946      0.968   0.0    0.887
Number_Travel_Lanes     0.971      0.993   0.0    0.941
Number_Park_Lanes       0.406      0.724   0.0    0.379
Number_Total_Lanes      0.971      0.993   0.0    0.926
StreetWidth_Min         0.971      0.993   0.0    0.941
StreetWidth_Max         0.971      0.993   0.0    0.941
StreetWidth_Irr         0.000      0.000   0.0    0.000
SHAPE_Length            1.000      1.000   0.0    1.000
LBoro                   1.000      1.000   0.0    1.000
RBoro                   1.000      1.000   0.0    1.000
FCC                     0.000      0.000   0.0    0.000
TrafDir                 0.998      0.999   0.0    0.998
TRUCK_ROUTE_TYPE        0.565      0.469   0.0    0.431
Snow_Priority           0.998      0.994   0.0    0.998
BikeLane                0.225      0.271   0.0    0.368
POSTED_SPEED
25.0    354855
30.0     31086
NaN      22913
50.0     12071
40.0      8578
20.0      5716
3

In [69]:
for c in ["StreetWidth_Irr", "FCC", "Number_Park_Lanes", "TRUCK_ROUTE_TYPE", "BikeLane"]:
    print("\n", c)
    print(lion[c].astype(str).str.strip().replace({"": "<blank>"}).value_counts(dropna=False).head(8))

print(auto_hourly2.drop_duplicates("segment_id")["lion_key_used"].value_counts())


 StreetWidth_Irr
StreetWidth_Irr
<blank>    243237
Name: count, dtype: int64

 FCC
FCC
<blank>    243228
10              9
Name: count, dtype: int64

 Number_Park_Lanes
Number_Park_Lanes
<blank>    123950
2           96272
1           23001
4              14
Name: count, dtype: int64

 TRUCK_ROUTE_TYPE
TRUCK_ROUTE_TYPE
<blank>    204454
2           23864
3           14676
1             243
Name: count, dtype: int64

 BikeLane
BikeLane
<blank>    214545
2           11170
1           10594
3            5715
4             411
6             309
11            221
9              90
Name: count, dtype: int64
lion_key_used
SegmentID    2421
LegacyID      281
spatial        94
none           47
Name: count, dtype: int64


In [70]:
av4 = av3.drop(columns=["StreetWidth_Irr", "FCC"])
matched = av4["lion_key_used"] != "none"

for c in ["Number_Park_Lanes", "TRUCK_ROUTE_TYPE", "BikeLane"]:
    v = pd.to_numeric(av4[c], errors="coerce")
    av4[c] = v.where(~matched | v.notna(), 0)      # blank -> 0 for matched rows only

assert len(av4) == 444140, "row count changed"

print(av4.groupby("lion_key_used")[["Number_Park_Lanes", "TRUCK_ROUTE_TYPE", "BikeLane"]]
        .agg(lambda s: s.notna().mean()).round(3))
for c in ["Number_Park_Lanes", "TRUCK_ROUTE_TYPE", "BikeLane"]:
    print(av4[c].value_counts(dropna=False).head(6))

av4.to_parquet(f"{OUT}/auto_hourly_v4.parquet", index=False)

               Number_Park_Lanes  TRUCK_ROUTE_TYPE  BikeLane
lion_key_used                                               
LegacyID                     1.0               1.0       1.0
SegmentID                    1.0               1.0       1.0
none                         0.0               0.0       0.0
spatial                      1.0               1.0       1.0
Number_Park_Lanes
2.0    241189
0.0    143036
1.0     53567
NaN      6348
Name: count, dtype: int64
TRUCK_ROUTE_TYPE
0.0    227860
2.0    164341
3.0     44004
NaN      6348
1.0      1587
Name: count, dtype: int64
BikeLane
0.0     320069
2.0      49649
1.0      34571
3.0      27589
NaN       6348
11.0      1912
Name: count, dtype: int64
